In [17]:
# Import necessary libraries
import numpy as np
import optuna
import os
import sqlite3
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import wandb

In [ ]:


# Generate a random dataset, just for demonstration purposes
X, y = make_classification(
    n_samples=1000, n_features=20, n_informative=15, n_redundant=5, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



In [ ]:


def train_a_model(trial_params):
    """
    THIS IS YOUR MODEL TRAINING CODE !!
    It depends on what you want to do; either
        1. train one model and return it's results
        2. do K-fold cross-validation, and return the mean result over all folds
        3. something else?
    """

    C = trial_params["C"]
    solver = trial_params["solver"]
    # Create a logistic regression model
    model = LogisticRegression(max_iter=1000, random_state=42, C=C, solver=solver)
    # OR do it like this:
    # Create a logistic regression model
    
    # Train the model
    model.fit(X_train, y_train)
    
    # Predict and calculate accuracy
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)

    results_dict = {
        "accuracy": accuracy,
        # add other metrics you want to log here
    }

    # log the results to WandB 
    wandb.log(results_dict)


    # in a deep learning model, you would probably log the losses and metrics of the model on each epoch
    # but here we just log the final accuracy of the model (as an example)
    # 
    # NOTE: if you logging on each epoch, then wandb will show the last epoch you did as the final result, not the 'best' epoch. 
    # you should keep track of the best epoch and log it separately using this code (which you can use each time you have a new best epoch):
    """
    for key in best_log_dict.keys():  # loop through the keys of the 'best epoch' log (a dictionary)
        wandb.define_metric(key, summary="none")

    # update the best log. This appears in the 'summary' tab in WandB (the table of results)
    wandb.run.summary.update(best_log_dict)
    """
    
    return results_dict


# Define the objective function for Optuna
def objective(trial):
    """
    An objective function for optuna to optimise. It takes a trial object as input and returns a value (or list of values) to be optimised (maximized or minimized).
    1. The trial object is used to suggest hyperparameters for the model. 
    2. The model is trained using these hyperparameters.
    3. The results of the model are reported to optuna.
    """

    # Suggest values for hyperparameters
    C = trial.suggest_float("C", 1e-4, 1e2)
    solver = trial.suggest_categorical("solver", ["liblinear", "lbfgs"])

    # sometimes you have conditional hyperparameters, this is how you could handle it
    if solver == "liblinear":
        trial.suggest_categorical("penalty", ["l1", "l2"])

    trial_params = trial.params   # trial.params is a dictionary of hyperparameters sampled by optuna
    print(f"Trial {trial.number}. Sampled parameters:", trial.params)

    # start a WandB log for this trial
    wandb.init(
        project="your_project_name",  # Replace with your project name
        config = trial_params   # tell WandB what the hyperparameters are on this trial, so that we can compare them in the web interface
    )

    #### TRAIN YOUR MODEL HERE ####
    results_dict = train_a_model(trial_params)

    # you must return only the values you want to optimise (maximise or minimise) in the objective function
    # do not return a dictionary. It should be either a single value, or a list of values
    
    results = [value for key, value in results_dict.items()]

    return results




##################################################

def main():
    studyName = "myOptunaExperiment"
    sampler = optuna.samplers.TPESampler(
                                        seed=42,
                                        n_startup_trials=50,   # how many totally random trials to run before the TPE sampler starts to use the results of previous trials to inform its sampling
                                        )
    
    # OPTIONAL: weights and biases logging
    wandb.login(key= "YOUR WandB API KEY HERE  !!!! " )   # NOTE: comment out this line if you don't have an account

    



    # Put the optuna study in a specific directory
    # NOTE: for Habrok, it might be necessary to define a sqlite storage path, especially if you want to run multiple jobs in parallel for the same study
    pathOptunaStudyTracker = os.path.join(os.getcwd(), studyName, "track_optuna.db")

    os.makedirs(studyName, exist_ok=True)  # Create the directory if it doesn't exist


    # Create an Optuna studye
    study = optuna.create_study(
                    #storage=pathOptunaStudyTracker,
                    study_name=studyName,  # the name of the current experiment
                    sampler=sampler,      # the sampler to use for the study. If not specified, the default sampler is used.
                    direction=["maximize", 'minimize'],  # in which direction to optimise. direction can also be a list of strings (['maximize', 'minimize', etc]
                    load_if_exists=True,  # helps if you want to continue a study that was interrupted or stopped
                    )   


    # Begin the experiment
    study.optimize(objective, n_trials=60)


    # Print the best hyperparameters and accuracy
    print("Best hyperparameters:", study.best_params)
    print("Best accuracy:", study.best_value)

main()

[I 2025-04-16 09:00:29,111] A new study created in memory with name: myOptunaExperiment
[I 2025-04-16 09:00:29,145] Trial 0 finished with value: 0.825 and parameters: {'C': 37.45407443072437, 'solver': 'liblinear', 'penalty': 'l1'}. Best is trial 0 with value: 0.825.
[I 2025-04-16 09:00:29,177] Trial 1 finished with value: 0.825 and parameters: {'C': 15.59953643416823, 'solver': 'lbfgs'}. Best is trial 0 with value: 0.825.
[I 2025-04-16 09:00:29,182] Trial 2 finished with value: 0.825 and parameters: {'C': 60.111541062819704, 'solver': 'liblinear', 'penalty': 'l1'}. Best is trial 0 with value: 0.825.
[I 2025-04-16 09:00:29,196] Trial 3 finished with value: 0.825 and parameters: {'C': 21.233989833916546, 'solver': 'lbfgs'}. Best is trial 0 with value: 0.825.
[I 2025-04-16 09:00:29,196] Trial 4 finished with value: 0.825 and parameters: {'C': 30.424293871729475, 'solver': 'liblinear', 'penalty': 'l2'}. Best is trial 0 with value: 0.825.
[I 2025-04-16 09:00:29,211] Trial 5 finished with v

Trial 0. Sampled parameters: {'C': 37.45407443072437, 'solver': 'liblinear', 'penalty': 'l1'}
Trial 1. Sampled parameters: {'C': 15.59953643416823, 'solver': 'lbfgs'}
Trial 2. Sampled parameters: {'C': 60.111541062819704, 'solver': 'liblinear', 'penalty': 'l1'}
Trial 3. Sampled parameters: {'C': 21.233989833916546, 'solver': 'lbfgs'}
Trial 4. Sampled parameters: {'C': 30.424293871729475, 'solver': 'liblinear', 'penalty': 'l2'}
Trial 5. Sampled parameters: {'C': 13.949472115818118, 'solver': 'lbfgs'}
Trial 6. Sampled parameters: {'C': 45.60705281470517, 'solver': 'liblinear', 'penalty': 'l2'}
Trial 7. Sampled parameters: {'C': 4.6451366269585, 'solver': 'liblinear', 'penalty': 'l2'}
Trial 8. Sampled parameters: {'C': 96.56320674425262, 'solver': 'liblinear', 'penalty': 'l2'}
Trial 9. Sampled parameters: {'C': 44.015305358710755, 'solver': 'lbfgs'}
Best hyperparameters: {'C': 37.45407443072437, 'solver': 'liblinear', 'penalty': 'l1'}
Best accuracy: 0.825
